# LangSmith Studio — step-by-step setup & debugging guide (with `uv`)

A hands-on walkthrough that takes you from zero to debugging a LangGraph agent in **LangSmith Studio**, using **`uv`** for environment and dependency management instead of `pip` + `venv`.

> **Why `uv`?** It replaces `pip`, `venv`, `pyenv`, and `poetry` with one fast Rust binary. `uv venv` creates virtualenvs in milliseconds, `uv pip install` resolves and installs an order of magnitude faster than pip, and `uv run` lets you execute scripts without manually activating the env. If you're new to it: <https://docs.astral.sh/uv/>.

> **⚠️ Run this locally, not on Colab.** Studio is a browser app at `https://smith.langchain.com/studio` that connects to a **local** LangGraph server you start with `langgraph dev`. Colab is a remote VM with no addressable localhost. The notebook is markdown + shell commands — you don't have to run any Python cells if you'd rather copy-paste into a terminal.

## What you'll build

Two projects, in order:

1. **`weather-agent/`** — a 30-line throwaway. Five minutes from `mkdir` to first run in Studio. Use this to learn the mechanics.
2. **`research-agent/`** — port the workshop's capstone. Use this to learn the debugging workflow that makes structured-state routing pay off.

## Prerequisites

- Python **3.11+** somewhere on your machine (or let `uv` install it for you — see §0)
- An **OpenAI API key** (`sk-...`)
- A **LangSmith API key** (`lsv2_...`) — free tier is fine; sign up at <https://smith.langchain.com>
- A terminal you can leave running while you click around in a browser

## Part 0 — Install `uv`

One-liner installer. Pick your platform:

In [ ]:
# macOS / Linux
curl -LsSf https://astral.sh/uv/install.sh | sh

# Windows (PowerShell)
# powershell -ExecutionPolicy ByPass -c "irm https://astral.sh/uv/install.ps1 | iex"

# Or via Homebrew / pipx if you already have them:
# brew install uv
# pipx install uv

Verify:

In [ ]:
uv --version
# uv 0.5.x or later — anything from late 2024 onward is fine

> **One-time setup.** `uv` doesn't need a global Python — it'll download the right interpreter on demand. If you want to pin one explicitly: `uv python install 3.11`.

## Part 1 — How Studio actually works

Studio doesn't host your graph. It's a frontend. The graph lives in a Python process you run locally with `langgraph dev`, and Studio talks to that process over HTTP. Mental model:

```
your laptop                                          smith.langchain.com
─────────────                                        ─────────────────────
langgraph dev   ←── HTTP ──→   Studio (browser tab) ←── auth ──  LangSmith
   │
   ├─ loads src/agent.py
   ├─ exposes /runs, /threads, /state on port 2024
   └─ holds checkpoints in RAM
```

When you click "Submit" in Studio, the browser POSTs to `http://127.0.0.1:2024/runs`. When you click "Re-run from here", it POSTs to a fork endpoint with the edited state. Everything runs in the Python process on your laptop — Studio is just a fancy GUI for the LangGraph server's HTTP API.

Three consequences worth internalizing:

- **Kill `langgraph dev` and Studio breaks.** The browser tab will show "no connection." Restart the server and refresh.
- **In-memory checkpoints don't survive restarts.** Saving a code file hot-reloads the graph and wipes threads. For multi-turn debugging across reloads, swap `InMemorySaver` for `SqliteSaver`.
- **Browser → localhost is allowed by CORS** because `langgraph dev` opens permissive headers. If you see CORS errors, make sure you opened Studio via the URL the CLI printed — that URL has the `?baseUrl=` param baked in.

## Part 2 — Project 1: the 5-minute weather agent

We'll scaffold a minimal project, start the server, open Studio, run a few invocations, and time-travel through the trace.

### 2.1 Create the project layout

In [ ]:
mkdir -p weather-agent/src
cd weather-agent

Final layout:

```
weather-agent/
├─ pyproject.toml
├─ .python-version
├─ uv.lock
├─ src/
│  └─ agent.py
├─ langgraph.json
└─ .env
```

### 2.2 Initialize the project with `uv`

`uv init` creates a `pyproject.toml`, pins the Python version, and seeds a virtualenv folder. We pass `--no-readme` and `--no-package` because we don't want sample boilerplate — just an env to install into.

In [ ]:
uv init --no-readme --no-package --python 3.11 .

This drops three files into the current dir:

- `pyproject.toml` — project metadata + dependency list
- `.python-version` — the Python version `uv` will use here (`3.11`)
- `main.py` — boilerplate sample; delete it: `rm main.py`

### 2.3 Add dependencies with `uv add`

`uv add` is the equivalent of `poetry add` — it (a) resolves the dep, (b) writes it into `pyproject.toml`, and (c) installs it into the project's virtualenv (creating `.venv/` if it doesn't exist). It's an order of magnitude faster than `pip install`.

In [ ]:
uv add \
    "langchain==1.2.15" \
    "langchain-openai==1.1.16" \
    "langchain-core==1.3.0" \
    "langgraph==1.1.9" \
    "langgraph-checkpoint-sqlite==3.0.3" \
    "openai==2.32.0" \
    "pydantic==2.13.3"

# The langgraph CLI itself, with the in-memory server extra
uv add "langgraph-cli[inmem]"

A `uv.lock` file appears — that's the resolved lockfile (commit it). The `.venv/` folder is also created automatically; don't commit that.

### 2.4 Create `src/agent.py`

Studio loads this file and looks for the variable named in `langgraph.json` (we'll wire that next). The variable must be a **compiled** LangGraph — i.e. the result of `builder.compile()` or `create_agent(...)`.

In [ ]:
%%writefile src/agent.py
from langchain.agents import create_agent
from langchain.tools import tool

@tool
def get_weather(city: str) -> str:
    """Get the weather for a city."""
    return f"Sunny, 19°C in {city}"

graph = create_agent(
    model="gpt-4o-mini",
    tools=[get_weather],
    system_prompt="You are a concise weather assistant.",
)

### 2.5 Create `langgraph.json`

Project manifest. The CLI reads it on startup. Format:

```
"<name shown in Studio>": "<relative_path>:<variable_name>"
```

You can register multiple graphs by adding more keys.

In [ ]:
%%writefile langgraph.json
{
  "dependencies": ["."],
  "graphs": {
    "weather_agent": "./src/agent.py:graph"
  },
  "env": ".env",
  "python_version": "3.11"
}

### 2.6 Create `.env`

Read by the CLI; values become environment variables for your graph. **Never commit it.**

In [ ]:
%%writefile .env
OPENAI_API_KEY=sk-replace-me
LANGSMITH_API_KEY=lsv2_replace-me
LANGSMITH_TRACING=true
LANGSMITH_PROJECT=studio-weather-demo

### 2.7 Start the server with `uv run`

`uv run <cmd>` automatically uses the project's virtualenv — no manual `source .venv/bin/activate` needed. It also re-syncs deps if `pyproject.toml` changed since the last run.

In [ ]:
uv run langgraph dev

You should see something like:

```
Welcome to LangGraph!

- 🚀 API:    http://127.0.0.1:2024
- 🎨 Studio: https://smith.langchain.com/studio/?baseUrl=http://127.0.0.1:2024
- 📚 API Docs: http://127.0.0.1:2024/docs

Watching for changes...
```

| URL                             | Purpose                                                    |
|---                              |---                                                         |
| `http://127.0.0.1:2024`         | Raw HTTP API — you usually don't visit this directly       |
| `…/studio/?baseUrl=…`           | The one you click — opens Studio in your browser           |
| `http://127.0.0.1:2024/docs`    | Auto-generated OpenAPI docs for the local server           |

**Ctrl-click the Studio URL.** Browser opens, authenticates against LangSmith using `LANGSMITH_API_KEY` from your `.env`, and the graph topology renders.

### 2.8 Common startup failures

| Symptom                                            | Fix                                                              |
|---                                                 |---                                                               |
| `Address already in use` on port 2024              | `uv run langgraph dev --port 2034` (then update the Studio URL)  |
| `ModuleNotFoundError: langgraph`                   | You forgot the `uv run` prefix; it activates the env for you      |
| `ImportError: graph` not found                     | Variable in `src/agent.py` doesn't match `langgraph.json`         |
| Studio loads forever, never shows the graph        | `baseUrl=` query param is missing or wrong — re-click the printed URL |
| Browser says CORS                                  | Same fix — the printed URL is the one that works                  |

Leave the server running in this terminal for the rest of Part 2.

## Part 3 — Tour the Studio UI

Studio has three panels.

### 3.1 Left panel — Graph

Renders your `StateGraph` topology. Boxes are nodes, lines are edges, dashed lines are conditional edges, curves are loop-backs. As a run executes, the active node pulses and edges light up as they're traversed.

What to do here:

- **Sanity-check topology.** If you see an unexpected node or a missing edge, your `add_node` / `add_edge` calls are wrong. Catch this before debugging anything else.
- **Click any node.** The centre panel filters to that node's invocations across the run.

### 3.2 Centre panel — Thread

A vertical timeline of every node execution for the currently-selected thread. Each entry is collapsible and shows:

- The input state delta the node received
- The output state delta the node returned
- Any `AIMessage` / `ToolMessage` content it emitted

Multiple runs on the same thread stack here, separated by checkpoint markers. Multi-turn conversations show as one growing thread; switching `thread_id` from the dropdown swaps the visible thread.

### 3.3 Right panel — three tabs

| Tab           | What it shows                                                            |
|---            |---                                                                       |
| **State**     | Full `StateSnapshot.values` at the currently-selected checkpoint, with a type-aware editor for every key |
| **Inputs**    | The input dict you submit to start a new run                             |
| **Configurable** | The `config={"configurable": {…}}` dict — most importantly `thread_id` |

The crucial button at the bottom of the State tab: **⟲ Re-run from here**. It forks the graph from the selected checkpoint, applies any state edits you've made, and runs forward.

## Part 4 — Three workflows you'll use constantly

### 4.1 Submit a run from scratch

1. Top of centre panel → **New thread** (Studio generates a `thread_id`).
2. Right panel → **Inputs** tab. For a `MessagesState` agent, the minimum input is:
   ```json
   { "messages": [{ "role": "user", "content": "What's the weather in Paris?" }] }
   ```
3. Click **Submit**.
4. Watch the graph panel: `__start__` → `agent` (the LLM node) → `tools` (calls `get_weather`) → `agent` (final answer) → `__end__`.
5. Centre panel populates with each node's I/O. Right-panel State tab now shows the final state.

### 4.2 Continue the same thread (multi-turn memory)

After §4.1 completes:

1. Stay on the same thread (don't click New thread).
2. Inputs tab → enter only the new turn:
   ```json
   { "messages": [{ "role": "user", "content": "And in Tokyo?" }] }
   ```
   Don't re-include prior messages — the checkpointer already has them.
3. Submit. The agent runs on top of the existing thread state and remembers Paris was just asked.

This is the GUI equivalent of:
```python
agent.invoke({"messages": [HumanMessage("And in Tokyo?")]},
             config={"configurable": {"thread_id": "<same id>"}})
```

### 4.3 Time travel — replay or fork from a past step

1. Centre panel → scroll to a past checkpoint (e.g. just before the second `agent` invocation).
2. Click that checkpoint's **⋯** menu → **Fork from here**. Studio selects it as the cursor.
3. (Optional) Right panel State tab → edit any field.
4. Click **⟲ Re-run from here**.
5. Studio creates a sibling branch and runs forward. The original is preserved.

Use this when you want to ask "what would have happened if X had been different at step 5?"

## Part 5 — Breakpoints (a.k.a. interrupts)

There are three ways to pause a graph for inspection or human approval. All of them surface the same way in Studio: the run pauses at the boundary, the right panel shows a **▶ Resume** button, and you can edit state before resuming.

### 5.1 Static interrupt — set in the Studio UI

Right-click any node in the graph panel → **Add interrupt before** or **Add interrupt after**. Studio sends a config flag to the server; the next run pauses at that boundary. Fastest way to step through a graph without touching code — useful when you don't yet know which node is misbehaving.

### 5.2 Compile-time interrupts — global, in code

```python
graph = builder.compile(
    checkpointer=InMemorySaver(),
    interrupt_before=["synthesizer"],
    interrupt_after=["researcher"],
)
```

These pause every run at the named boundaries. Studio surfaces them identically. Useful for HITL pipelines where the pause point is fixed (e.g. "always pause before sending the email").

### 5.3 Dynamic interrupt — `interrupt()` inside a node

For real human-in-the-loop where the *payload* matters:

```python
from langgraph.types import interrupt, Command

def review_gate(state):
    decision = interrupt({
        "evidence_count": len(state["evidence"]),
        "sources":        state["sources"],
    })
    if decision.startswith("redo:"):
        return {"sub_questions": [decision.split(":", 1)[1]],
                "critic_sufficient": False}
    return {}
```

When the graph hits this node, Studio shows the dict you passed to `interrupt(...)` in the right panel and waits. Type a value (e.g. `"approve"` or `"redo:some new query"`) and click **Resume**. The server returns your value as the result of `interrupt(...)` and the node continues.

## Part 6 — Project 2: port the workshop capstone

Stop the previous `langgraph dev` (Ctrl-C) and create a fresh project.

### 6.1 Initialize the second project

In [ ]:
cd ..                                 # back out of weather-agent/
mkdir -p research-agent/src
cd research-agent

uv init --no-readme --no-package --python 3.11 .
rm main.py

# Same dep set
uv add \
    "langchain==1.2.15" \
    "langchain-openai==1.1.16" \
    "langchain-core==1.3.0" \
    "langgraph==1.1.9" \
    "langgraph-checkpoint-sqlite==3.0.3" \
    "openai==2.32.0" \
    "pydantic==2.13.3"
uv add "langgraph-cli[inmem]"

> **💡 Faster way for repeat setups.** Once you've got a known-good `pyproject.toml` and `uv.lock`, just `cp` them into the new project folder and run `uv sync`. That installs the exact locked versions in one shot — much faster than re-resolving with `uv add`.

### 6.2 Create `src/research_agent.py`

This is the workshop's capstone (§8.1 – §8.7) flattened into one file. The only addition at the bottom is `graph = research_agent` — Studio looks for the top-level `graph` variable.

In [ ]:
%%writefile src/research_agent.py
import os, time, operator
from typing import Annotated
from typing_extensions import TypedDict
from contextlib import contextmanager

from langchain.chat_models import init_chat_model
from langchain.messages import AnyMessage, HumanMessage, AIMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver
from pydantic import BaseModel, Field


# ── State schema ───────────────────────────────────────────────────────
class ResearchState(TypedDict):
    messages:          Annotated[list[AnyMessage], add_messages]
    research_question: str
    sub_questions:     Annotated[list[str], operator.add]
    evidence:          Annotated[list[dict], operator.add]
    sources:           Annotated[list[str], operator.add]
    critic_sufficient: bool
    critic_reasoning:  str
    critic_next_query: str
    iterations:        int
    max_iterations:    int
    error_count:       int
    final_answer:      str


# ── Search backend (local corpus — no Tavily key required) ─────────────
LOCAL_CORPUS = [
    {"title": "LangGraph v1 release notes",
     "url":   "docs.langchain.com/oss/python/langgraph/overview",
     "content": "LangGraph 1.0 stabilized the StateGraph API, renamed MemorySaver to "
                "InMemorySaver, and introduced durable Command/interrupt types. The "
                "langgraph.prebuilt.create_react_agent factory was deprecated in favor "
                "of langchain.agents.create_agent."},
    {"title": "create_agent — LangChain reference",
     "url":   "docs.langchain.com/oss/python/langchain/agents",
     "content": "create_agent is the canonical entry-point for tool-using agents in "
                "LangChain v1."},
    {"title": "Reducers in LangGraph",
     "url":   "docs.langchain.com/oss/python/langgraph/graph-api",
     "content": "A reducer merges a node\'s returned partial update into state. Default "
                "is overwrite; add_messages appends messages by id."},
    {"title": "Checkpointers and threads",
     "url":   "docs.langchain.com/oss/python/langgraph/persistence",
     "content": "A checkpointer makes a graph durable and replayable. Threads isolate "
                "state by thread_id; get_state_history walks every checkpoint."},
    {"title": "LangSmith Studio",
     "url":   "docs.langchain.com/oss/python/langgraph/studio",
     "content": "LangSmith Studio is a browser IDE for debugging LangGraph apps. "
                "Visualizes the graph, shows state at every step, lets you edit state "
                "and re-run from any checkpoint."},
    {"title": "Streaming modes",
     "url":   "docs.langchain.com/oss/python/langgraph/streaming",
     "content": "stream_mode=values yields full snapshots; updates yields per-node "
                "deltas; messages yields LLM token chunks."},
]

def search(query: str, k: int = 4) -> list[dict]:
    q = set(query.lower().split())
    def score(doc):
        text = (doc["title"] + " " + doc["content"]).lower()
        return sum(1 for w in q if w in text)
    ranked = sorted(LOCAL_CORPUS, key=score, reverse=True)
    return [d for d in ranked if score(d) > 0][:k] or ranked[:k]


# ── Pydantic schemas ──────────────────────────────────────────────────
class Plan(BaseModel):
    sub_questions: list[str] = Field(min_length=2, max_length=5)

class Critique(BaseModel):
    sufficient: bool
    reasoning:  str
    next_query: str = ""


# ── Instrumentation (prints to the langgraph dev console) ─────────────
run_log: list[dict] = []

@contextmanager
def timed(node_name: str):
    t0 = time.perf_counter()
    try:
        yield
    finally:
        ms = round((time.perf_counter() - t0) * 1000, 1)
        run_log.append({"node": node_name, "ms": ms})
        print(f"[{node_name}] {ms} ms")


# ── Nodes ──────────────────────────────────────────────────────────────
model = init_chat_model("gpt-4o-mini", temperature=0)
planner_llm = model.with_structured_output(Plan)
critic_llm  = model.with_structured_output(Critique)

def planner(state: ResearchState) -> dict:
    with timed("planner"):
        plan = planner_llm.invoke([
            SystemMessage("Break the research question into 2–5 atomic sub-questions."),
            HumanMessage(state["research_question"]),
        ])
    return {"sub_questions": plan.sub_questions,
            "messages": [AIMessage(f"Plan → {plan.sub_questions}")]}

def researcher(state: ResearchState) -> dict:
    it = state.get("iterations", 0)
    with timed(f"researcher#{it + 1}"):
        query = (state.get("critic_next_query")
                 or (state["sub_questions"][it] if it < len(state["sub_questions"])
                     else state["research_question"]))
        try:
            hits = search(query)
            evidence = [{"query": query, "title": h["title"],
                         "snippet": h["content"][:400]} for h in hits]
            sources  = [h["url"] for h in hits if h.get("url")]
            return {"evidence": evidence, "sources": sources,
                    "iterations": it + 1, "critic_next_query": "",
                    "messages": [AIMessage(f"Searched {query!r} → {len(evidence)} hits")]}
        except Exception as e:
            return {"error_count": state.get("error_count", 0) + 1,
                    "messages":    [AIMessage(f"Search error: {e}")]}

def critic(state: ResearchState) -> dict:
    with timed(f"critic#{state.get('iterations', 0)}"):
        summary = "\n".join(f"- {e['title']}: {e['snippet'][:120]}"
                             for e in state["evidence"][-8:])
        v = critic_llm.invoke([
            SystemMessage("Judge whether evidence answers the question. "
                          "If not, propose ONE specific follow-up query."),
            HumanMessage(f"Question: {state['research_question']}\n\nEvidence:\n{summary}"),
        ])
    return {"critic_sufficient": v.sufficient,
            "critic_reasoning":  v.reasoning,
            "critic_next_query": v.next_query,
            "messages": [AIMessage(f"Critic → sufficient={v.sufficient}; {v.reasoning}")]}

def synthesizer(state: ResearchState) -> dict:
    with timed("synthesizer"):
        bullets = "\n".join(f"- {e['title']}: {e['snippet'][:200]}"
                              for e in state["evidence"])
        cites   = "\n".join(f"[{i+1}] {u}" for i, u in enumerate(state["sources"]))
        ai = model.invoke([
            SystemMessage("Synthesize a concise cited answer. Use [n] footnotes."),
            HumanMessage(f"Question: {state['research_question']}\n\n"
                         f"Evidence:\n{bullets}\n\nSources:\n{cites}"),
        ])
    return {"final_answer": ai.content, "messages": [AIMessage(ai.content)]}

def error_handler(state: ResearchState) -> dict:
    with timed("error_handler"):
        msg = f"Research failed after {state.get('error_count', 0)} errors."
    return {"final_answer": msg, "messages": [AIMessage(msg)]}


# ── Routing reads typed state fields only — never message strings ──────
def route_after_critic(state: ResearchState) -> str:
    if state.get("error_count", 0) >= 2:                  return "error"
    if state["iterations"] >= state["max_iterations"]:    return "synth"
    if state.get("critic_sufficient", False):             return "synth"
    return "research"


# ── Assemble & compile ────────────────────────────────────────────────
b = StateGraph(ResearchState)
b.add_node("planner",       planner)
b.add_node("researcher",    researcher)
b.add_node("critic",        critic)
b.add_node("synthesizer",   synthesizer)
b.add_node("error_handler", error_handler)

b.add_edge(START,        "planner")
b.add_edge("planner",    "researcher")
b.add_edge("researcher", "critic")
b.add_conditional_edges("critic", route_after_critic, {
    "research": "researcher",
    "synth":    "synthesizer",
    "error":    "error_handler",
})
b.add_edge("synthesizer",   END)
b.add_edge("error_handler", END)

# Studio looks for a top-level variable named `graph`.
graph = b.compile(checkpointer=InMemorySaver())

> **🏭 Note about checkpointers in deployed graphs.** When you deploy this graph to the LangGraph Platform (LangSmith's managed runtime), remove the `checkpointer=InMemorySaver()` from `.compile()` — the platform injects its own. The local `langgraph dev` server happily uses whatever you pass.

### 6.3 `langgraph.json`

In [ ]:
%%writefile langgraph.json
{
  "dependencies": ["."],
  "graphs": {
    "research_agent": "./src/research_agent.py:graph"
  },
  "env": ".env",
  "python_version": "3.11"
}

### 6.4 `.env`

In [ ]:
%%writefile .env
OPENAI_API_KEY=sk-replace-me
LANGSMITH_API_KEY=lsv2_replace-me
LANGSMITH_TRACING=true
LANGSMITH_PROJECT=studio-research-demo

### 6.5 Run the server

In [ ]:
uv run langgraph dev

Open the printed Studio URL.

### 6.6 Submit your first capstone run

Right panel → **Inputs** tab → paste this initial state:

```json
{
  "messages": [],
  "research_question": "What are the key differences between LangGraph v0 and v1?",
  "sub_questions": [],
  "evidence": [],
  "sources": [],
  "critic_sufficient": false,
  "critic_reasoning": "",
  "critic_next_query": "",
  "iterations": 0,
  "max_iterations": 3,
  "error_count": 0,
  "final_answer": ""
}
```

Click **Submit**. You should see, in the graph panel:

`planner` → `researcher` → `critic` → (loop back to `researcher`) → `critic` → `synthesizer` → `END`

The terminal running `uv run langgraph dev` prints the per-node timings from the `timed()` instrumentation.

## Part 7 — The killer demo: debugging structured-state routing

This is the payoff of the capstone's design. Because `route_after_critic` reads only typed state fields (`critic_sufficient`, `iterations`, `error_count`) and never parses message content, you can debug the router by **editing one boolean**.

Try this in Studio:

1. After §6.6 completes, find the centre-panel entry for **`critic` at iteration 1**.
2. Click it. The right panel switches to that checkpoint.
3. **State** tab → find `critic_sufficient`. It's `false` (which is why the loop ran another researcher iteration).
4. **Toggle it to `true`.**
5. Click **⟲ Re-run from here**.
6. Watch the graph panel. The agent skips the second research loop entirely and jumps straight to `synthesizer`.

You just changed the agent's routing decision *without touching code or prompts*. That's only possible because the router reads typed state.

Compare this to a string-parsing router (`if "sufficient=True" in last_message.content`): you'd have to find and edit the *message*, hope your regex still matches, and re-run blindly.

Now try the inverse:

1. Find a checkpoint where `critic_sufficient` is `true`.
2. Toggle it to `false`.
3. Re-run. The agent does another research loop.

The original runs are preserved as sibling branches — Studio's tree view lets you flip between them.

## Part 8 — A real failure walkthrough

You'll eventually hit a bug where the agent returns `"Research failed after 2 errors"` even though the question is simple. Here's the diagnostic playbook in Studio.

**Step 1 — Graph panel: topology check.** Confirm the edges match your mental model. Missing edge? Unreachable node? Fix that first.

**Step 2 — Centre panel: find the anomaly.** Scroll the timeline. You see `researcher` ran twice followed immediately by `error_handler`. Suspicious — searches *looked* like they succeeded.

**Step 3 — Click `researcher` (iteration 2) → State tab at its input checkpoint.** You see `critic_next_query: ""` and `sub_questions[1]` is also `""`. Iteration 2 is searching for the empty string, returning no hits, treating "no hits" as failure, bumping `error_count`.

**Step 4 — Click the preceding `critic` (iteration 1) → State tab at its output checkpoint.** You see `critic_sufficient: false`, `critic_reasoning: "Need more on …"`, and crucially `critic_next_query: ""`. The critic *intended* to propose a follow-up but emitted an empty string. **The bug is the critic's prompt** — not the router, not the researcher.

**Step 5 — Validate the hypothesis with one click.** At that same checkpoint, type `"LangGraph v1 breaking changes"` into the `critic_next_query` editor. **⟲ Re-run from here**. Now `researcher#2` runs with a real query, returns 4 hits, `critic#2` accepts, `synthesizer` produces an answer. Confirmed.

**Step 6 — Share the trace.** Top of the thread → **Share**. Studio gives you a `https://smith.langchain.com/public/...` URL. Paste into your PR or ticket. Reviewer opens it and sees the exact pathological state without running anything.

**Step 7 — Fix the prompt.** Edit the critic's `SystemMessage` in `src/research_agent.py` to require a non-empty `next_query` whenever `sufficient=False`. Save the file. `langgraph dev` hot-reloads. Re-submit the original input. Bug is gone.

This loop — *edit state in Studio → confirm the hypothesis → fix the code → hot-reload* — is the workflow Studio is built for. It's faster than print-statement debugging by a factor of 10× on a real graph.

## Part 9 — Smell → cause cheat sheet

Pattern-match these when scanning a misbehaving run.

| Smell in Studio                          | Likely cause                                                              |
|---                                       |---                                                                        |
| Same node 3+ times consecutively         | Missing or too-loose termination condition; check your `recursion_limit`  |
| Graph has unreachable nodes              | Forgot an edge, or router's `path_map` doesn't cover a return value       |
| State key grows unbounded across runs    | Wrong reducer (`operator.add` where you wanted overwrite)                 |
| Routing oscillates between two nodes     | Router reads a field that's written *after* it runs                       |
| Tool node returns no messages            | Tool threw an exception you swallowed; check the tool's I/O panel         |
| Router silently takes the "error" branch | Upstream node returned an incomplete state dict — check its output        |
| Same input gives different graphs        | Non-deterministic node (random / time / external API) without a seed      |
| Run ends without `synthesizer` firing    | Conditional edge map is missing the success key                           |

## Part 10 — Make checkpoints survive restarts

`InMemorySaver` is fine for the first 10 minutes. For real debugging — where you want to come back to a thread tomorrow — swap it for SQLite. Edit the bottom of `src/research_agent.py`:

```python
import os
from langgraph.checkpoint.sqlite import SqliteSaver

if os.getenv("LANGGRAPH_DEPLOYED"):
    # Platform injects its own checkpointer; don't pass one.
    graph = b.compile()
else:
    # Local dev — persistent SQLite so threads survive hot-reloads and restarts.
    _sqlite_ck = SqliteSaver.from_conn_string("dev.db").__enter__()
    graph = b.compile(checkpointer=_sqlite_ck)
```

Now `dev.db` accumulates checkpoints. Add it to `.gitignore` — it has real conversation data. To start fresh: `rm dev.db`.

> **🏭 Production option.** For Postgres, use `langgraph.checkpoint.postgres.PostgresSaver`. Same pattern. It supports concurrent writers, which `SqliteSaver` does not.

## Part 11 — Studio vs other observability tools

| Need                          | Notebook               | Studio              | LangSmith trace view  |
|---                            |---                     |---                  |---                    |
| Topology check                | `draw_mermaid_png()`   | ✅ live graph        | ❌                     |
| Live state inspection         | print statements       | ✅ per-step viewer   | ✅ post-hoc            |
| Time travel                   | checkpoint config      | ✅ one click         | ❌                     |
| Edit state & rerun            | ❌                     | ✅                   | ❌                     |
| Hot-reload on file save       | ❌                     | ✅                   | ❌                     |
| Share a trace                 | ❌                     | ✅ public URL        | ✅                     |
| Add to dataset for eval       | ❌                     | ✅                   | ✅                     |
| Long-term archive             | ❌                     | ❌ (RAM/SQLite)      | ✅                     |

**Rule of thumb:** notebook for *development*, Studio for *debugging* a specific run, LangSmith for *production observability* over time.

## Part 12 — `uv` tips & gotchas

### 12.1 The commands you'll actually use

| Command                            | What it does                                                   |
|---                                 |---                                                             |
| `uv init <name>`                   | Create a new project (`pyproject.toml` + `.python-version`)    |
| `uv add <pkg>`                     | Add + install a runtime dependency                             |
| `uv add --dev <pkg>`               | Add + install a dev-only dep (test/lint tooling)              |
| `uv remove <pkg>`                  | Uninstall + remove from `pyproject.toml`                       |
| `uv sync`                          | Install exactly what's in `uv.lock` (CI / fresh checkout)     |
| `uv run <cmd>`                     | Run a command inside the project's virtualenv                  |
| `uv run --with <pkg> python ...`   | One-off run with an extra ephemeral dep                        |
| `uv lock --upgrade`                | Re-resolve everything to latest compatible versions            |
| `uv tree`                          | Show the dependency tree                                       |
| `uv pip list`                      | List installed packages (pip-compatible)                       |

### 12.2 Studio-specific gotchas

| Gotcha                                                       | Workaround                                                              |
|---                                                           |---                                                                       |
| Studio shows no graph                                        | Reopen via the URL printed by `uv run langgraph dev` — the `?baseUrl=` param matters |
| `ImportError` on `graph` not found                           | Variable name in `src/*.py` must match the value in `langgraph.json`     |
| `InMemorySaver` resets on every code save                    | Switch to `SqliteSaver.from_conn_string("dev.db")`                        |
| `recursion_limit` exceeded                                   | Bump it: Configurable tab → `recursion_limit: 50` (or pass in `config`)   |
| Slow tool calls time out                                     | `uv run langgraph dev --http-timeout 300`                                 |
| Graph won't reload after editing `src/`                      | `langgraph dev` was started with `--no-reload`, or you edited a file outside `src/` |
| Want a different port                                        | `uv run langgraph dev --port 2034` (then update the `baseUrl=` in the URL) |
| Studio asks for auth on every refresh                        | Make sure `LANGSMITH_API_KEY` is in `.env` and `LANGSMITH_TRACING=true`   |
| Multiple graphs registered, only one shows                   | Use the dropdown at the top of the graph panel to switch                  |
| Deployed graph rejects `checkpointer=...`                    | Remove from `.compile()` — the platform injects its own. Gate locally with `if not LANGGRAPH_DEPLOYED:` |
| `uv add` complains about Python version                       | `uv python install 3.11` then re-run                                      |

## Part 13 — What to commit, what not to

Add this to `.gitignore` in every Studio project:

In [ ]:
%%writefile .gitignore
.env
*.db
.langgraph_api/
.venv/
__pycache__/

**Commit:**

- `src/`
- `pyproject.toml` — your dependency declarations
- `uv.lock` — pinned, reproducible install (the whole point of using `uv`)
- `.python-version`
- `langgraph.json`
- `.env.example` (a redacted template — same keys, dummy values)

**Don't commit:**

- `.env` — has real keys
- `*.db` — has real conversation data
- `.langgraph_api/` — server cache
- `.venv/` — your virtualenv

That's everything. The natural workflow:

1. `uv run langgraph dev` running in one terminal.
2. Editor open on `src/your_agent.py`.
3. Studio in a browser tab.
4. Edit code → save → Studio hot-reloads → test → if broken, click a checkpoint, edit state, **Re-run from here** → confirm hypothesis → fix code → save → repeat.

That tight loop is what makes Studio worth wiring up for every serious LangGraph project.